# Lección 16 - Despliegue de Agentes Escalables con Microsoft Foundry

En este cuaderno construyes un **agente de soporte al cliente listo para producción** para la compañía ficticia **Contoso**. A diferencia de las lecciones anteriores, el objetivo aquí no es el ciclo de razonamiento del agente — es todo lo que lo rodea *alrededor* que hace que un agente sea seguro para ejecutar a escala:

1. **Llamada a herramientas** — consultas de pedidos y creación de tickets.
2. **RAG** — respuestas políticas de una base de conocimiento.
3. **Memoria** — recordar al cliente durante múltiples interacciones.
4. **Enrutamiento del modelo** — enviar solicitudes simples a un modelo pequeño, y las complejas a un modelo grande.
5. **Caché de respuestas** — servir preguntas repetidas sin llamar al modelo.
6. **Aprobación humana** — reembolsos por encima de un umbral se detienen para aprobación.
7. **Puerta de evaluación** — un conjunto de pruebas offline que bloquea un lanzamiento malo.
8. **Observabilidad** — rastreo con OpenTelemetry alrededor de cada solicitud.

Cada sección es autónoma y ejecutable. Lee cada línea — las primitivas de producción se mantienen deliberadamente pequeñas.


## Configuración

Antes de ejecutar este cuaderno, asegúrate de tener:

1. **Un proyecto Microsoft Foundry** con un modelo de chat desplegado (por ejemplo, `gpt-4.1-mini`).
2. **Sesión iniciada con Azure CLI** — ejecuta `az login` en tu terminal.
3. **Establecidas las variables de entorno necesarias:**
   - `AZURE_AI_PROJECT_ENDPOINT` — el endpoint de tu proyecto Microsoft Foundry.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — el nombre de tu modelo desplegado.

La sección RAG utiliza **Azure AI Search** cuando están configurados `AZURE_SEARCH_SERVICE_ENDPOINT` y `AZURE_SEARCH_API_KEY`, y recurre a una búsqueda en memoria para que el cuaderno funcione sin un recurso Search.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import re
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential(),
)
print("Foundry client ready.")

## 1. Herramientas

Las herramientas de producción realizan trabajo real contra sistemas reales. Aquí simulamos una base de datos de pedidos y un sistema de tickets con funciones simples de Python. El decorador `@tool` las expone al agente.

Nótese que `issue_refund` utiliza `approval_mode="always_require"` para reembolsos por encima de un umbral — esta es la primitiva de humano en el bucle que implementamos más adelante.


In [ ]:
# Simulated backend systems (in production these are API calls behind scoped identities).
ORDERS = {
    "A1001": {"status": "shipped", "total": 42.00, "eta": "2 days"},
    "A1002": {"status": "processing", "total": 128.50, "eta": "5 days"},
    "A1003": {"status": "delivered", "total": 19.99, "eta": "delivered"},
}
TICKETS: list[dict] = []
REFUND_APPROVAL_THRESHOLD = 50.0


@tool(approval_mode="never_require")
def get_order_status(order_id: Annotated[str, "The customer's order ID, e.g. A1001"]) -> str:
    """Look up the status of a customer order."""
    order = ORDERS.get(order_id.upper())
    if not order:
        return f"No order found with ID {order_id}."
    return (
        f"Order {order_id.upper()}: status={order['status']}, "
        f"total=${order['total']:.2f}, eta={order['eta']}."
    )


@tool(approval_mode="never_require")
def open_ticket(
    subject: Annotated[str, "Short subject line for the support ticket"],
    details: Annotated[str, "Full description of the customer's issue"],
) -> str:
    """Open a support ticket for issues that need human follow-up."""
    ticket_id = f"T{1000 + len(TICKETS) + 1}"
    TICKETS.append({"id": ticket_id, "subject": subject, "details": details})
    return f"Ticket {ticket_id} opened: {subject}"


def refund_needs_approval(amount: float) -> bool:
    """Refunds above the threshold require a human approver."""
    return amount > REFUND_APPROVAL_THRESHOLD


@tool(approval_mode="always_require")
def issue_refund(
    order_id: Annotated[str, "The order to refund"],
    amount: Annotated[float, "Refund amount in USD"],
) -> str:
    """Issue a refund. Execution pauses for human approval before it runs."""
    return f"Refund of ${amount:.2f} issued for order {order_id.upper()}."


print("Tools defined.")

## 2. RAG — Base de Conocimiento de Políticas

Las preguntas sobre políticas ("¿cuál es tu plazo de devolución?") deben responderse desde una fuente autorizada, no desde la memoria del modelo. Envolvemos una pequeña base de conocimiento como una herramienta de búsqueda.

En producción esto es **Azure AI Search**; aquí proporcionamos una búsqueda por palabras clave en memoria para que el cuaderno se ejecute en cualquier lugar, y cambiamos automáticamente a Azure AI Search cuando las variables de entorno están presentes.


In [ ]:
KNOWLEDGE_BASE = {
    "returns": "Contoso accepts returns within 30 days of delivery for a full refund. Items must be unused and in original packaging.",
    "shipping": "Standard shipping takes 3-5 business days. Express shipping (1-2 days) is available at checkout for an extra fee.",
    "warranty": "All Contoso electronics carry a 12-month limited warranty covering manufacturing defects.",
    "refund_policy": "Refunds are processed to the original payment method within 5 business days of approval. Refunds over $50 require a supervisor's approval.",
}


def _in_memory_search(query: str) -> str:
    q = query.lower()
    hits = [text for key, text in KNOWLEDGE_BASE.items() if key.replace("_", " ") in q or key in q]
    if not hits:
        # crude keyword fallback so the tool still returns something useful
        hits = [text for text in KNOWLEDGE_BASE.values() if any(w in text.lower() for w in q.split())]
    return "\n".join(hits) if hits else "No matching policy found."


def _azure_search(query: str) -> str:
    from azure.core.credentials import AzureKeyCredential
    from azure.search.documents import SearchClient

    client = SearchClient(
        endpoint=os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"],
        index_name=os.getenv("AZURE_SEARCH_INDEX_NAME", "contoso-policies"),
        credential=AzureKeyCredential(os.environ["AZURE_SEARCH_API_KEY"]),
    )
    results = client.search(search_text=query, top=3)
    return "\n".join(r.get("content", "") for r in results) or "No matching policy found."


USE_AZURE_SEARCH = bool(os.getenv("AZURE_SEARCH_SERVICE_ENDPOINT") and os.getenv("AZURE_SEARCH_API_KEY"))


@tool(approval_mode="never_require")
def search_policies(query: Annotated[str, "The policy question to look up"]) -> str:
    """Search Contoso support policies to answer customer questions."""
    if USE_AZURE_SEARCH:
        return _azure_search(query)
    return _in_memory_search(query)


print(f"RAG ready. Using {'Azure AI Search' if USE_AZURE_SEARCH else 'in-memory search'}.")

## 3. Memoria

Un agente de soporte que olvida con quién está hablando es un mal agente de soporte. Guardamos una pequeña tienda de perfiles por cliente e inyectamos un resumen breve en las instrucciones del agente. En producción, esto es un servicio de memoria (ver Lección 13); aquí, un diccionario hace visible el patrón.


In [ ]:
CUSTOMER_MEMORY: dict[str, dict] = {
    "cust-42": {"name": "Dana", "tier": "enterprise", "recent_order": "A1002"},
    "cust-99": {"name": "Sam", "tier": "standard", "recent_order": "A1003"},
}


def memory_context(customer_id: str) -> str:
    profile = CUSTOMER_MEMORY.get(customer_id)
    if not profile:
        return "This is a new customer with no history."
    return (
        f"Customer {profile['name']} ({profile['tier']} tier). "
        f"Most recent order: {profile['recent_order']}."
    )


print(memory_context("cust-42"))

## 4 y 5. Enrutamiento de modelos y caché de respuestas

Dos palancas de costo conectadas a un solo manejador de solicitudes:

- **Enrutamiento**: un clasificador heurístico económico decide si una solicitud necesita el modelo pequeño o el grande.
- **Caché**: preguntas repetidas normalizadas se sirven directamente desde una caché sin llamada al modelo.

El clasificador aquí es intencionalmente simple. En producción lo validarías con el tráfico y podrías reemplazarlo con el Enrutador de Modelos de Foundry.


In [ ]:
SMALL_MODEL = os.getenv("AZURE_AI_SMALL_MODEL", model)   # e.g. gpt-4.1-mini
LARGE_MODEL = os.getenv("AZURE_AI_LARGE_MODEL", model)   # e.g. gpt-4.1

response_cache: dict[str, str] = {}
route_counters = {"small": 0, "large": 0, "cache": 0}


def normalize(query: str) -> str:
    return re.sub(r"\s+", " ", query.lower().strip())


COMPLEX_SIGNALS = ("refund", "cancel", "complaint", "escalate", "broken", "wrong", "why")


def is_simple(query: str) -> bool:
    """Route complex or high-stakes requests to the large model; everything else to the small one."""
    q = query.lower()
    if any(signal in q for signal in COMPLEX_SIGNALS):
        return False
    return len(q.split()) <= 20


def choose_model(query: str) -> str:
    return SMALL_MODEL if is_simple(query) else LARGE_MODEL


print(f"Small model: {SMALL_MODEL} | Large model: {LARGE_MODEL}")

## 6 y 8. El Agente, Aprobación Humana y Observabilidad

Ahora ensamblamos el agente a partir de las herramientas anteriores y envolvemos cada solicitud en un span de OpenTelemetry. La función `handle_support_request` es el manejador de solicitudes en producción: caché → ruta → traza → ejecución → caché.

La aprobación humana es gestionada por el framework: debido a que `issue_refund` tiene `approval_mode="always_require"`, la ejecución se pausa y muestra una solicitud de aprobación que resolvemos explícitamente.


In [ ]:
# Tracing: use the Agent Framework tracer if available, else a no-op so the notebook runs anywhere.
try:
    from agent_framework.observability import get_tracer
    tracer = get_tracer()
except Exception:  # observability extras not installed
    from contextlib import contextmanager

    class _NoopSpan:
        def set_attribute(self, *_args, **_kwargs):
            pass

    class _NoopTracer:
        @contextmanager
        def start_as_current_span(self, _name):
            yield _NoopSpan()

    tracer = _NoopTracer()


SUPPORT_INSTRUCTIONS = (
    "You are Contoso's customer support agent. Be concise, friendly, and accurate. "
    "Use search_policies for policy questions, get_order_status for orders, "
    "open_ticket when a human needs to follow up, and issue_refund for refunds. "
    "Never invent policy details."
)

support_agent = provider.as_agent(
    name="ContosoSupportAgent",
    instructions=SUPPORT_INSTRUCTIONS,
    tools=[get_order_status, open_ticket, search_policies, issue_refund],
)


async def handle_support_request(query: str, customer_id: str) -> str:
    # 1. Serve from cache when we can.
    key = normalize(query)
    if key in response_cache:
        route_counters["cache"] += 1
        return response_cache[key]

    # 2. Route by complexity to control cost.
    chosen_model = choose_model(query)
    route_counters["small" if chosen_model == SMALL_MODEL else "large"] += 1

    # 3. Add per-customer memory to the prompt.
    context = memory_context(customer_id)
    prompt = f"[Customer context: {context}]\n\n{query}"

    # 4. Run inside a trace span for observability.
    with tracer.start_as_current_span("support_request") as span:
        span.set_attribute("customer.id", customer_id)
        span.set_attribute("routed.model", chosen_model)
        response = await support_agent.run(prompt, model=chosen_model)

    text = response.text
    response_cache[key] = text
    return text


print("Support agent assembled.")

In [ ]:
# Try a few requests. The first is simple (small model), the second is a refund (large model + approval path).
print(await handle_support_request("What is your return window?", "cust-99"))
print("---")
print(await handle_support_request("Where is my order A1002?", "cust-42"))
print("---")
# Repeat the first question -> served from cache.
print(await handle_support_request("What is your return window?", "cust-99"))
print("---")
print("Routing counters:", route_counters)

## 7. Puerta de Evaluación

Esta es la puerta de lanzamiento de la lección: un conjunto de prueba fuera de línea evalúa al agente, y el despliegue solo procede si la tasa de aprobación supera un umbral. El evaluador aquí es una verificación simple de superposición de palabras clave para mantener el cuaderno autosuficiente; en producción usarías un LLM como juez o un evaluador de marco (ver Lección 10).


In [ ]:
TEST_CASES = [
    {"input": "How long do I have to return an item?", "expected": ["30 days", "refund"]},
    {"input": "How fast is standard shipping?", "expected": ["3-5", "business days"]},
    {"input": "What is the status of order A1001?", "expected": ["shipped", "A1001"]},
    {"input": "Do your electronics have a warranty?", "expected": ["12-month", "warranty"]},
]


def score_response(actual: str, expected_keywords: list[str]) -> float:
    actual_l = actual.lower()
    hits = sum(1 for kw in expected_keywords if kw.lower() in actual_l)
    return hits / len(expected_keywords)


async def evaluation_gate(test_cases: list[dict], threshold: float = 0.8) -> bool:
    passed = 0
    for case in test_cases:
        result = await support_agent.run(case["input"])
        s = score_response(result.text, case["expected"])
        status = "PASS" if s >= 0.5 else "FAIL"
        print(f"[{status}] {case['input']}  (score={s:.0%})")
        if s >= 0.5:
            passed += 1
    pass_rate = passed / len(test_cases)
    print(f"\nEvaluation pass rate: {pass_rate:.0%} (gate: {threshold:.0%})")
    return pass_rate >= threshold


gate_passed = await evaluation_gate(TEST_CASES, threshold=0.8)
print("\nDeploy allowed:" , gate_passed)

## Poniéndolo todo junto: un lanzamiento simulado

La celda a continuación muestra todo el ciclo que la lección describe: ejecutar la puerta de evaluación y solo "desplegar" si pasa. Este es el patrón que ejecutarías en CI antes de promover una versión del agente al Servicio de Agentes Foundry.


In [ ]:
async def release(test_cases: list[dict]) -> None:
    print("Running pre-deployment evaluation gate...\n")
    if await evaluation_gate(test_cases, threshold=0.8):
        print("\n✅ Gate passed — promoting agent version to the Foundry Agent Service.")
    else:
        print("\n❌ Gate failed — release blocked. Fix the agent and re-run.")


await release(TEST_CASES)

## Resumen

Has ensamblado un agente de soporte al cliente listo para producción con todas las preocupaciones operativas integradas:

- **Herramientas, RAG y memoria** otorgan capacidad y contexto al agente.
- **Enrutamiento y almacenamiento en caché de modelos** mantienen la latencia y el costo bajo control.
- **Aprobación humana** protege acciones de alto riesgo como reembolsos grandes.
- **La puerta de evaluación** bloquea malas versiones antes de que se publiquen.
- **Trazabilidad** hace cada solicitud observable.

### Desafío

Extiende este agente para:

1. **Soportar múltiples modelos** — agrega un tercer nivel de "razonamiento" y enruta las escaladas/quejas hacia él.
2. **Agregar puertas de evaluación** — amplía `TEST_CASES` para incluir escenarios de aprobación de reembolso y confirma que la puerta detecta regresiones.
3. **Agregar enrutamiento consciente de costos** — sigue un costo estimado por solicitud (pequeña vs grande vs caché) e imprime un reporte de costos después de un lote de consultas mixtas.

En la siguiente lección tomas el camino contrario y ejecutas un agente completamente en tu máquina con Microsoft Foundry Local y Qwen.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Descargo de responsabilidad**:
Este documento ha sido traducido utilizando el servicio de traducción automática [Co-op Translator](https://github.com/Azure/co-op-translator). Aunque nos esforzamos por la precisión, tenga en cuenta que las traducciones automatizadas pueden contener errores o inexactitudes. El documento original en su idioma nativo debe considerarse la fuente autorizada. Para información crítica, se recomienda una traducción profesional humana. No somos responsables de cualquier malentendido o interpretación errónea que surja del uso de esta traducción.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
